In [6]:
import pandas as pd
import numpy as np 
import os 
from pathlib import Path

In [7]:
df = pd.read_csv('../data/metadata_final_cloud.csv')
df_filter = df[(df['dataset_id'] == 'jpr_dados') | (df['dataset_id'] == 'physionet') | (df['dataset_id'] == 'cq500_dados')]

In [8]:


# Define base paths
jpr_base = Path(r"D:\Backup\Mestrado\dados\JPR\dataset_jpr_train\dataset_jpr_train")
physionet_base = Path(r"D:\Backup\Mestrado\dados\Physionet\physionet_scans\ct_scans")

# 1. Pre-index JPR directories (since they are split across folders 1, 2, 3)
jpr_paths = {}
for sub in ['1', '2', '3']:
    p = jpr_base / sub
    if p.exists():
        for folder in p.iterdir():
            if folder.is_dir():
                jpr_paths[folder.name] = folder.resolve().as_posix()

# 2. Function to resolve local path based on dataset_id
def get_local_path(row):
    pid = str(row['patient_id']).strip()
    dataset = row['dataset_id']
    
    if dataset == 'physionet':
        # Formats 49 -> 049.nii
        filename = f"{pid.zfill(3)}.nii"
        file_path = physionet_base / filename
        return file_path.resolve().as_posix() if file_path.exists() else None
        
    elif dataset == 'jpr_dados':
        # Formats 1171 -> 001171
        formatted_id = pid.zfill(6)
        return jpr_paths.get(formatted_id, None)
    
    return None

# 3. Apply to df_filter
df_filter['local_Path'] = df_filter.apply(get_local_path, axis=1)

# Check the result
print(df_filter[['dataset_id', 'patient_id', 'local_Path']].head())

     dataset_id patient_id                                         local_Path
2628  jpr_dados       1171  D:/Backup/Mestrado/dados/JPR/dataset_jpr_train...
2629  jpr_dados       2125  D:/Backup/Mestrado/dados/JPR/dataset_jpr_train...
2630  jpr_dados       1442  D:/Backup/Mestrado/dados/JPR/dataset_jpr_train...
2631  jpr_dados        873  D:/Backup/Mestrado/dados/JPR/dataset_jpr_train...
2632  jpr_dados       2332  D:/Backup/Mestrado/dados/JPR/dataset_jpr_train...


In [10]:
df_filter.to_csv('../data/metadata_local.csv', index = False)